In [2]:

import subprocess
subprocess.run(['pip', 'install', 'openpyxl', '-q'], capture_output=True)

from openpyxl import Workbook
from openpyxl.styles import (Font, PatternFill, Alignment, Border, Side,
                              GradientFill)
from openpyxl.utils import get_column_letter
import os

wb = Workbook()

# ─── Foglio principale ────────────────────────────────────────────────────────
ws = wb.active
ws.title = 'Obiettivi Segreti'

# Palette colori
COLOR_HEADER_BG  = '1a2744'
COLOR_HEADER_FG  = '00ff88'
COLOR_SECT_BG    = '0d1424'
COLOR_SECT_FG    = 'ffffff'
COLOR_ROW_ALT    = '111827'
COLOR_ROW_NORM   = '0d1424'

FAZIONE_COLORS = {
    'Iran':                   ('22c55e', '🇮🇷'),
    'Coalizione Occidentale': ('3b82f6', '🇺🇸'),
    'Russia':                 ('ef4444', '🇷🇺'),
    'Cina':                   ('f59e0b', '🇨🇳'),
    'Unione Europea':         ('8b5cf6', '🇪🇺'),
}

thin = Side(style='thin', color='1e3a5f')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

header_font = Font(name='Consolas', bold=True, color=COLOR_HEADER_FG, size=10)
header_fill = PatternFill('solid', fgColor=COLOR_HEADER_BG)

# Colonne
columns = [
    ('Codice',             18),
    ('Fazione',            28),
    ('Nome Obiettivo',     30),
    ('Descrizione',        65),
    ('Punti Fine Partita', 10),
    ('Difficoltà',         12),
    ('Tipo Condizione',    18),
    ('Campo Condizione',   22),
    ('Operatore',          12),
    ('Valore Soglia',      14),
    ('Note Condizione',    55),
    ('Attivo',              8),
]

# Header riga 1
for col_idx, (label, width) in enumerate(columns, start=1):
    cell = ws.cell(row=1, column=col_idx, value=label)
    cell.font = header_font
    cell.fill = header_fill
    cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
    cell.border = border
    ws.column_dimensions[get_column_letter(col_idx)].width = width

ws.row_dimensions[1].height = 28
ws.freeze_panes = 'A2'

# Dati: 15 obiettivi reali + 3 righe di esempio
obiettivi = [
    # Iran
    ('OBJ_IRAN_01', 'Iran', 'Soglia Nucleare',
     'Raggiungi il penultimo spazio del Tracciato Nucleare senza che la Coalizione effettui un attacco militare diretto contro i tuoi siti nucleari.',
     8, 'difficile', 'tracciato', 'nucleare', '>=', 14,
     'Verificare che non siano avvenuti attacchi militari ai siti nucleari iraniani durante la partita.', 'true'),
    ('OBJ_IRAN_02', 'Iran', "Asse della Resistenza",
     'Controlla almeno 3 nazioni tra Libano, Siria, Iraq e Yemen contemporaneamente alla fine della partita.',
     6, 'media', 'territorio', 'controllo', '>=', 3,
     'Contare quante tra Libano, Siria, Iraq, Yemen risultano controllate da Iran alla fine.', 'true'),
    ('OBJ_IRAN_03', 'Iran', 'Blocco dello Stretto',
     'Mantieni almeno 2 unità navali nello Stretto di Hormuz per 3 turni consecutivi senza subire perdite.',
     5, 'media', 'territorio', 'Stretto di Hormuz', '>=', 2,
     'Verificare presenza di 2+ unità navali nello Stretto di Hormuz per 3 turni consecutivi.', 'true'),
    # Coalizione
    ('OBJ_COAL_01', 'Coalizione Occidentale', 'Smantellamento Nucleare',
     "Riporta il Tracciato Nucleare iraniano al livello iniziale attraverso azioni diplomatiche o militari.",
     8, 'difficile', 'tracciato', 'nucleare', '<=', 1,
     "Il tracciato nucleare deve essere tornato al valore di inizio partita (1).", 'true'),
    ('OBJ_COAL_02', 'Coalizione Occidentale', 'Cambio di Regime',
     "Porta il Tracciato Sanzioni/Stabilità Economica dell'Iran al livello critico, causando proteste interne diffuse.",
     7, 'difficile', 'tracciato', 'sanzioni', '>=', 9,
     "Il tracciato sanzioni deve raggiungere il livello critico (9-10). Verificare stabilità Iran bassa.", 'true'),
    ('OBJ_COAL_03', 'Coalizione Occidentale', 'Sicurezza Energetica',
     "Mantieni il controllo o l'influenza dominante su almeno 2 nazioni produttrici di petrolio (Arabia Saudita, Iraq, Kuwait) fino alla fine della partita.",
     5, 'media', 'territorio', 'controllo', '>=', 2,
     "Arabia Saudita, Iraq, Kuwait: almeno 2 devono essere sotto influenza Coalizione alla fine.", 'true'),
    # Russia
    ('OBJ_RUS_01', 'Russia', 'Mediatore Indispensabile',
     "Usa l'abilità Veto almeno 3 volte durante la partita e mantieni influenza in almeno 2 nazioni controllate da fazioni diverse.",
     6, 'media', 'carta', 'veto', '>=', 3,
     "Contare le carte Veto giocate (min 3) e verificare influenza in 2+ nazioni di fazioni diverse.", 'true'),
    ('OBJ_RUS_02', 'Russia', 'Contratti Militari',
     "Vendi sistemi d'arma (gioca carte specifiche) ad almeno 3 nazioni diverse durante la partita.",
     5, 'facile', 'carta', 'contratto_militare', '>=', 3,
     "Contare le carte di vendita sistemi d'arma giocate verso nazioni diverse (min 3 nazioni).", 'true'),
    ('OBJ_RUS_03', 'Russia', 'Base Mediterranea',
     "Controlla la Siria e mantieni almeno 1 unità navale nel Mediterraneo Orientale alla fine della partita.",
     7, 'media', 'territorio', 'Siria', '==', 1,
     "Siria controllata da Russia + almeno 1 unità navale in Mediterraneo Orientale alla fine.", 'true'),
    # Cina
    ('OBJ_CINA_01', 'Cina', 'Via della Seta Energetica',
     "Piazza influenza BRI in almeno 4 nazioni diverse, creando una catena di connessione commerciale.",
     7, 'difficile', 'territorio', 'influenza_bri', '>=', 4,
     "Contare le nazioni con influenza BRI cinese al termine della partita (minimo 4).", 'true'),
    ('OBJ_CINA_02', 'Cina', 'Stabilità Commerciale',
     "Termina la partita senza che il Tracciato Tensione Globale (DEFCON) superi mai il livello critico (scenda sotto 4).",
     6, 'media', 'tracciato', 'defcon', '>=', 4,
     "Il DEFCON non deve mai scendere sotto 4 durante tutta la partita. Verificare nel log turni.", 'true'),
    ('OBJ_CINA_03', 'Cina', 'Partner Silenzioso',
     "Mantieni relazioni commerciali (influenza) sia con l'Iran che con almeno una nazione della Coalizione alla fine della partita.",
     5, 'facile', 'territorio', 'influenza_mista', '>=', 2,
     "Cina deve avere influenza sia su Iran che su almeno una nazione Coalizione alla fine.", 'true'),
    # Europa
    ('OBJ_EU_01', 'Unione Europea', 'Diplomazia Multilaterale',
     "Fai approvare almeno 2 Risoluzioni ONU durante la partita senza che vengano bloccate dal veto russo o cinese.",
     6, 'media', 'carta', 'risoluzione_onu', '>=', 2,
     "Contare le risoluzioni ONU approvate (non bloccate da veto Russia/Cina): minimo 2.", 'true'),
    ('OBJ_EU_02', 'Unione Europea', 'Crisi Umanitaria Evitata',
     "Nessuna nazione adiacente all'Europa (Turchia, Libano, Siria) deve avere stabilità inferiore a 2 alla fine della partita.",
     7, 'difficile', 'territorio', 'stabilita_adiacente', '>=', 2,
     "Turchia, Libano, Siria: tutti e 3 devono avere stabilità >= 2 alla fine. Verificare nel game_state.", 'true'),
    ('OBJ_EU_03', 'Unione Europea', 'Accordo Nucleare Rinnovato',
     "Negozia un nuovo accordo JCPOA: il Tracciato Nucleare deve essere nella metà inferiore e il Tracciato Sanzioni non deve essere al massimo.",
     8, 'difficile', 'tracciato', 'nucleare', '<=', 7,
     "Nucleare <= 7 E Sanzioni < 10 alla fine della partita. Entrambe le condizioni devono essere soddisfatte.", 'true'),
    # Righe di esempio per nuovi obiettivi
    ('OBJ_IRAN_04', 'Iran', '← ESEMPIO: inserisci qui il tuo obiettivo Iran',
     'Descrizione completa della condizione di vittoria segreta per Iran...',
     6, 'media', 'tracciato', 'nucleare', '>=', 10,
     'Come verificare manualmente questa condizione a fine partita.', 'true'),
    ('OBJ_COAL_04', 'Coalizione Occidentale', '← ESEMPIO Coalizione',
     "Isola diplomaticamente l'Iran ottenendo l'appoggio di almeno 4 nazioni al Consiglio di Sicurezza.",
     7, 'media', 'carta', 'appoggio_cs', '>=', 4,
     "Contare le nazioni con voto favorevole alla Coalizione al CS ONU.", 'true'),
    ('OBJ_EU_04', 'Unione Europea', '← ESEMPIO Europa',
     "Coordina almeno 3 pacchetti di aiuti umanitari durante la crisi senza rompere le relazioni diplomatiche con Iran.",
     5, 'facile', 'carta', 'aiuto_umanitario', '>=', 3,
     "Contare le carte aiuto umanitario giocate dall'Europa (min 3).", 'true'),
]

for row_idx, obj in enumerate(obiettivi, start=2):
    faz = obj[1]
    is_example = '← ESEMPIO' in str(obj[2])
    faz_color = FAZIONE_COLORS.get(faz, ('8899aa', '🌐'))[0]

    for col_idx, value in enumerate(obj, start=1):
        cell = ws.cell(row=row_idx, column=col_idx, value=value)
        cell.border = border
        cell.alignment = Alignment(vertical='top', wrap_text=(col_idx in (4, 11)))

        if is_example:
            cell.fill = PatternFill('solid', fgColor='1a2030')
            cell.font = Font(name='Consolas', color='445566', size=9, italic=True)
        elif col_idx == 1:  # Codice
            cell.fill = PatternFill('solid', fgColor='0d1424')
            cell.font = Font(name='Consolas', color='8b5cf6', size=9, bold=True)
        elif col_idx == 2:  # Fazione
            cell.fill = PatternFill('solid', fgColor=faz_color + '33')
            cell.font = Font(name='Consolas', color=faz_color, size=9, bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='top')
        elif col_idx == 5:  # Punti
            cell.fill = PatternFill('solid', fgColor='1a1200')
            cell.font = Font(name='Consolas', color='f59e0b', size=10, bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='top')
        elif col_idx == 6:  # Difficoltà
            diff_colors = {'facile': '22c55e', 'media': 'f59e0b', 'difficile': 'ef4444'}
            dc = diff_colors.get(str(value), '8899aa')
            cell.fill = PatternFill('solid', fgColor=dc + '20')
            cell.font = Font(name='Consolas', color=dc, size=9, bold=True)
            cell.alignment = Alignment(horizontal='center', vertical='top')
        elif col_idx in (3,):  # Nome
            cell.fill = PatternFill('solid', fgColor='111827' if row_idx % 2 == 0 else '0d1424')
            cell.font = Font(name='Consolas', color='ffffff', size=9, bold=True)
        elif col_idx == 4:  # Descrizione
            cell.fill = PatternFill('solid', fgColor='0a0e1a')
            cell.font = Font(name='Consolas', color='ccddee', size=9)
        elif col_idx == 11:  # Note
            cell.fill = PatternFill('solid', fgColor='0a0e1a')
            cell.font = Font(name='Consolas', color='8899aa', size=8)
        else:
            cell.fill = PatternFill('solid', fgColor='111827' if row_idx % 2 == 0 else '0d1424')
            cell.font = Font(name='Consolas', color='aabbcc', size=9)

    ws.row_dimensions[row_idx].height = 45 if obj[1] in ('Descrizione',) else 40

# Altezze righe più alte per descrizioni
for r in range(2, len(obiettivi)+2):
    ws.row_dimensions[r].height = 42

# ─── Foglio Istruzioni ────────────────────────────────────────────────────────
wi = wb.create_sheet('📋 Istruzioni')
wi.sheet_view.showGridLines = False
wi.column_dimensions['A'].width = 30
wi.column_dimensions['B'].width = 72
wi.column_dimensions['C'].width = 15

istr_data = [
    ('LINEA ROSSA — Template Obiettivi Segreti', '', ''),
    ('', '', ''),
    ('CAMPO', 'VALORI AMMESSI / DESCRIZIONE', 'OBBLIGATORIO'),
    ('Codice', 'Formato: OBJ_FAZIONE_NN  (es. OBJ_IRAN_04, OBJ_EU_05)', 'SÌ'),
    ('Fazione', 'Iran | Coalizione Occidentale | Russia | Cina | Unione Europea | Neutrale', 'SÌ'),
    ('Nome Obiettivo', 'Testo libero (max 60 caratteri consigliati)', 'SÌ'),
    ('Descrizione', 'Testo narrativo completo della condizione di vittoria segreta', 'SÌ'),
    ('Punti Fine Partita', 'Intero 1-15. Linea guida: 5=facile · 6-7=media · 8=difficile', 'SÌ'),
    ('Difficoltà', 'facile | media | difficile  (tutto minuscolo)', 'SÌ'),
    ('Tipo Condizione', 'tracciato | territorio | carta | manuale', 'No'),
    ('Campo Condizione', 'nucleare | sanzioni | defcon | opinione | controllo | carta_type | ecc.', 'No'),
    ('Operatore', '>=  |  <=  |  ==', 'No'),
    ('Valore Soglia', 'Numero intero (soglia numerica). Lasciare vuoto per condizioni testuali.', 'No'),
    ('Note Condizione', 'Come verificare manualmente la condizione a fine partita. Testo libero.', 'No'),
    ('Attivo', 'true | false  (default: true)', 'No'),
    ('', '', ''),
    ('FAZIONI VALIDE', '', ''),
    ('Iran', 'Fazione iraniana — obiettivi offensivi/nucleari', ''),
    ('Coalizione Occidentale', 'USA + alleati — obiettivi deterrenza/sanzioni', ''),
    ('Russia', 'Federazione Russa — obiettivi influenza/commercio', ''),
    ('Cina', "Repubblica Popolare Cinese — obiettivi BRI/diplomazia", ''),
    ('Unione Europea', 'UE — obiettivi pace/umanitari', ''),
    ('', '', ''),
    ('NOTE IMPORTANTI', '', ''),
    ('Codici già usati', 'OBJ_IRAN_01..03 · OBJ_COAL_01..03 · OBJ_RUS_01..03 · OBJ_CINA_01..03 · OBJ_EU_01..03', ''),
    ('Upsert sicuro', 'Il caricamento usa upsert su obj_id: aggiornare un obiettivo esistente è sicuro.', ''),
    ('Condizioni manuali', 'Per obiettivi verificati a mano, lasciare vuoti Tipo/Campo/Operatore/Valore.', ''),
    ('Codici progressivi', 'Usa OBJ_IRAN_04, OBJ_EU_04 ecc. per i nuovi obiettivi aggiuntivi.', ''),
]

title_font = Font(name='Consolas', bold=True, color='00ff88', size=14)
header2_font = Font(name='Consolas', bold=True, color='ffffff', size=10)
sect_font = Font(name='Consolas', bold=True, color='8b5cf6', size=10)
norm_font = Font(name='Consolas', color='aabbcc', size=9)
note_font = Font(name='Consolas', color='8899aa', size=9)

for r, (a, b, c) in enumerate(istr_data, start=1):
    ca = wi.cell(row=r, column=1, value=a)
    cb = wi.cell(row=r, column=2, value=b)
    cc = wi.cell(row=r, column=3, value=c)

    for cell in (ca, cb, cc):
        cell.fill = PatternFill('solid', fgColor='0a0e1a')
        cell.alignment = Alignment(vertical='top', wrap_text=True)
        cell.border = Border(bottom=Side(style='thin', color='1e3a5f'))

    if r == 1:
        ca.font = title_font
        wi.merge_cells(f'A1:C1')
        wi.row_dimensions[1].height = 32
    elif a == 'CAMPO':
        for cell in (ca, cb, cc):
            cell.font = Font(name='Consolas', bold=True, color='f59e0b', size=10)
            cell.fill = PatternFill('solid', fgColor='1a1200')
        wi.row_dimensions[r].height = 22
    elif a in ('FAZIONI VALIDE', 'NOTE IMPORTANTI'):
        ca.font = sect_font
        wi.row_dimensions[r].height = 22
    elif a in ('Codice', 'Fazione', 'Nome Obiettivo', 'Descrizione', 'Punti Fine Partita',
               'Difficoltà', 'Tipo Condizione', 'Campo Condizione', 'Operatore',
               'Valore Soglia', 'Note Condizione', 'Attivo'):
        ca.font = Font(name='Consolas', bold=True, color='8b5cf6', size=9)
        cb.font = norm_font
        cc.font = Font(name='Consolas', color='22c55e' if c == 'SÌ' else '8899aa', size=9)
        wi.row_dimensions[r].height = 20
    elif a in ('Iran', 'Coalizione Occidentale', 'Russia', 'Cina', 'Unione Europea'):
        faz_c = FAZIONE_COLORS.get(a, ('8899aa', ''))[0]
        ca.font = Font(name='Consolas', bold=True, color=faz_c, size=9)
        cb.font = note_font
        wi.row_dimensions[r].height = 18
    elif a.startswith('Codici') or a.startswith('Upsert') or a.startswith('Condiz') or a.startswith('Codici prog'):
        ca.font = Font(name='Consolas', bold=True, color='f59e0b', size=9)
        cb.font = note_font
        wi.row_dimensions[r].height = 18
    else:
        ca.font = norm_font
        cb.font = norm_font

# ─── Salvataggio ──────────────────────────────────────────────────────────────
out_path = '/workspace/linea_rossa_tracker/public/template_obiettivi_segreti.xlsx'
wb.save(out_path)
print(f"✅ Template salvato: {out_path}")
print(f"   Dimensione: {os.path.getsize(out_path):,} bytes")
print(f"   Righe dati: {len(obiettivi)}")


✅ Template salvato: /workspace/linea_rossa_tracker/public/template_obiettivi_segreti.xlsx
   Dimensione: 10,639 bytes
   Righe dati: 18
